<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/02_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part-2 Dataset preparation for model fine-tuning

In [ ]:
# Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

In [ ]:
# Clone or pull GitHub repository

repo_url = "https://github.com/Al-Amin95/PromptInjectionDetectionSystem.git"
branch_name = "data-analysis"
repo_path = Path("/content/PromptInjectionDetectionSystem")

if repo_path.exists():
    print("Repository already exists in Colab.")
    print("Pulling latest changes from GitHub...")

    os.chdir(repo_path)
    !git fetch origin
    !git checkout {branch_name}
    !git pull origin {branch_name}

else:
    print("Repository not found in Colab.")
    print("Cloning repository from GitHub...")

    os.chdir("/content")
    !git clone -b {branch_name} {repo_url}
    os.chdir(repo_path)

print("\nCurrent working directory:")
print(os.getcwd())

print("\nCurrent Git branch:")
!git branch

print("\nProject files:")
!ls -lh

print("\nNotebooks folder:")
!ls -lh notebooks

print("\nRaw data folder:")
!ls -lh data/raw

In [ ]:
# Set project paths

PROJECT_DIR = Path("/content/PromptInjectionDetectionSystem")

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/USW_Dissertation/PromptInjectionDetectionSystem")
DRIVE_PROCESSED_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "processed"
DRIVE_TABLES_DIR = DRIVE_PROJECT_DIR / "results" / "tables"
DRIVE_NOTEBOOKS_DIR = DRIVE_PROJECT_DIR / "notebooks"

folders_to_create = [
    PROCESSED_DATA_DIR,
    TABLES_DIR,
    DRIVE_PROCESSED_DATA_DIR,
    DRIVE_TABLES_DIR,
    DRIVE_NOTEBOOKS_DIR
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

print("Project paths are ready.")

print("\nGitHub project paths:")
print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

print("\nGoogle Drive backup paths:")
print("DRIVE_PROJECT_DIR:", DRIVE_PROJECT_DIR)
print("DRIVE_PROCESSED_DATA_DIR:", DRIVE_PROCESSED_DATA_DIR)
print("DRIVE_TABLES_DIR:", DRIVE_TABLES_DIR)
print("DRIVE_NOTEBOOKS_DIR:", DRIVE_NOTEBOOKS_DIR)

In [ ]:
# Check raw dataset files exist

TRAIN_FILE = RAW_DATA_DIR / "train-00000-of-00001-9564e8b05b4757ab.parquet"
TEST_FILE = RAW_DATA_DIR / "test-00000-of-00001-701d16158af87368.parquet"

print("Train file exists:", TRAIN_FILE.exists())
print("Test file exists:", TEST_FILE.exists())

print("\nTrain file path:")
print(TRAIN_FILE)

print("\nTest file path:")
print(TEST_FILE)

print("\nFiles inside raw data folder:")
for file in RAW_DATA_DIR.iterdir():
    print(file.name)

In [ ]:
#Load raw train and test datasets

train_df = pd.read_parquet(TRAIN_FILE)
test_df = pd.read_parquet(TEST_FILE)

print("Raw train dataset loaded successfully.")
print("Raw test dataset loaded successfully.")

print("\nTrain dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

print("\nTrain dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

In [ ]:
# Import required libraries for dataset preparation

import os
import re
import shutil
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

In [ ]:
# Add readable label_name column

label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

train_df["label_name"] = train_df["label"].map(label_mapping)
test_df["label_name"] = test_df["label"].map(label_mapping)

print("label_name column added successfully.")

print("\nLabel mapping:")
for label, label_name in label_mapping.items():
    print(f"{label} = {label_name}")

print("\nTrain dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows from train dataset:")
display(train_df.head())

print("\nFirst 5 rows from test dataset:")
display(test_df.head())

In [ ]:
# Verify label values and label mapping

print("Unique labels in train dataset:")
print(sorted(train_df["label"].unique()))

print("\nUnique labels in test dataset:")
print(sorted(test_df["label"].unique()))

print("\nUnique label names in train dataset:")
print(sorted(train_df["label_name"].unique()))

print("\nUnique label names in test dataset:")
print(sorted(test_df["label_name"].unique()))

print("\nTrain label counts:")
print(train_df["label"].value_counts().sort_index())

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

print("\nTrain label_name counts:")
print(train_df["label_name"].value_counts())

print("\nTest label_name counts:")
print(test_df["label_name"].value_counts())

# Check whether any label_name value is missing after mapping
print("\nMissing label_name values in train dataset:", train_df["label_name"].isnull().sum())
print("Missing label_name values in test dataset:", test_df["label_name"].isnull().sum())

In [ ]:
# Confirm raw train/test size and columns

raw_dataset_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [train_df.shape[0], test_df.shape[0]],
    "columns": [train_df.shape[1], test_df.shape[1]],
    "column_names": [train_df.columns.tolist(), test_df.columns.tolist()]
})

print("Raw dataset summary:")
display(raw_dataset_summary)

print("\nTrain dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

print("\nTrain dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

# Cleaning Rules- after Part 1 Findings

1. The dataset has no missing values.
2. The dataset has no empty text values.
3. The dataset has no whitespace-only text values.
4. The dataset has no duplicate full rows.
5. The dataset has no duplicate text values.
6. The dataset has no conflicting labels.
7. The dataset has no train/test text overlap.
8. Very short prompts were inspected and appeared valid.
9. Very long prompts were inspected and appeared to be valid `INJECTION` examples.
10. `INJECTION` prompts were generally longer than `SAFE` prompts.
11. `INJECTION` prompts contained important instruction-style and command-style patterns, including:

    1. `ignore`
    2. `forget`
    3. `previous`
    4. `instruction`
    5. `instructions`
    6. `prompt`
    7. `output`
    8. `print`
    9. `act as`

## 11.2 Cleaning Decision

1. Aggressive text cleaning is not required.
2. Only light text cleaning will be applied.
3. The original text will be preserved before cleaning.
4. A new cleaned text column will be created instead of replacing the original text column.

## 11.3 Cleaning Rules

1. Preserve the original `text` column.
2. Create a new `original_text` column to store the unchanged raw prompt.
3. Create a new `clean_text` column for the cleaned version.
4. Remove leading and trailing spaces.
5. Replace multiple spaces with a single space.
6. Replace repeated new lines, tabs, or extra whitespace with a single space.
7. Keep punctuation and symbols.
8. Keep capitalisation.
9. Keep short prompts.
10. Keep long prompts.
11. Keep injection-related words and phrases.
12. Apply the same cleaning rule to train, validation, and test data.

## 11.4 What Will Not Be Removed

1. Stopwords will not be removed.
2. Punctuation will not be removed.
3. Special characters will not be removed aggressively.
4. Text will not be converted fully to lowercase.
5. Numbers will not be removed.
6. Command-style words will not be removed.
7. Prompt-injection-related terms such as `ignore`, `forget`, `previous`, `instruction`, `prompt`, `output`, and `print` will not be removed.
8. Prompts will not be removed only because they are short.
9. Prompts will not be removed only because they are long.

## 11.5 Reason for Light Cleaning

1. Prompt injection signals may appear in punctuation, symbols, casing, and command-style phrases.
2. Aggressive cleaning could remove useful attack indicators.
3. Transformer models can process natural text directly, so heavy preprocessing is not necessary.
4. The dataset was already clean in Part 1, so only minimal formatting cleanup is needed.
5. Preserving the original prompt structure is important for fine-tuning DistilBERT, RoBERTa, and SecBERT-style models.

## 11.6 Final Cleaning Strategy

1. The project will use light cleaning only.
2. The original prompt text will be preserved.
3. The cleaned text will be used for model fine-tuning.
4. The original test set will remain untouched in terms of split membership.
5. The same cleaning function will be applied consistently to train, validation, and test data.


In [ ]:
# Preserve original text column

train_df["original_text"] = train_df["text"].copy()
test_df["original_text"] = test_df["text"].copy()

print("Original text column preserved successfully.")

print("\nTrain dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows from train dataset:")
display(train_df[["text", "original_text", "label", "label_name"]].head())

print("\nFirst 5 rows from test dataset:")
display(test_df[["text", "original_text", "label", "label_name"]].head())

# Check whether text and original_text are exactly the same
train_same_check = (train_df["text"] == train_df["original_text"]).all()
test_same_check = (test_df["text"] == test_df["original_text"]).all()

print("\nTrain text and original_text are exactly same:", train_same_check)
print("Test text and original_text are exactly same:", test_same_check)

In [ ]:
# Define light text-cleaning function

def light_clean_text(text):
    """
    Apply light text cleaning while preserving prompt-injection signals.
    """

    # Convert text to string in case any unexpected non-string value appears
    text = str(text)

    # Remove leading and trailing spaces
    text = text.strip()

    # Replace tabs, new lines, and repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text)

    return text


print("Light text-cleaning function defined successfully.")

# Quick test examples
example_texts = [
    "   Ignore all previous instructions.   ",
    "Hello\n\nworld\tthis is a test.",
    "Generate SQL code to access a database.",
    "Visit https://example.com and print output."
]

print("\nCleaning examples:")
for example in example_texts:
    print("\nOriginal:")
    print(example)
    print("Cleaned:")
    print(light_clean_text(example))

In [ ]:
# Apply light text cleaning

train_df["clean_text"] = train_df["original_text"].apply(light_clean_text)
test_df["clean_text"] = test_df["original_text"].apply(light_clean_text)

print("Light text cleaning applied successfully.")

print("\nTrain dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows from train dataset:")
display(train_df[["original_text", "clean_text", "label", "label_name"]].head())

print("\nFirst 5 rows from test dataset:")
display(test_df[["original_text", "clean_text", "label", "label_name"]].head())

In [ ]:
# Verify clean_text column

print("Checking clean_text column in train dataset:")
print("clean_text exists in train:", "clean_text" in train_df.columns)

print("\nChecking clean_text column in test dataset:")
print("clean_text exists in test:", "clean_text" in test_df.columns)

print("\nMissing clean_text values in train dataset:")
print(train_df["clean_text"].isnull().sum())

print("\nMissing clean_text values in test dataset:")
print(test_df["clean_text"].isnull().sum())

print("\nEmpty clean_text values in train dataset:")
print((train_df["clean_text"] == "").sum())

print("\nEmpty clean_text values in test dataset:")
print((test_df["clean_text"] == "").sum())

print("\nTrain clean_text data type:")
print(train_df["clean_text"].dtype)

print("\nTest clean_text data type:")
print(test_df["clean_text"].dtype)

print("\nFirst 5 clean_text values from train dataset:")
display(train_df[["clean_text", "label", "label_name"]].head())

print("\nFirst 5 clean_text values from test dataset:")
display(test_df[["clean_text", "label", "label_name"]].head())

In [ ]:
# Compare original_text and clean_text

train_df["text_changed_after_cleaning"] = train_df["original_text"] != train_df["clean_text"]
test_df["text_changed_after_cleaning"] = test_df["original_text"] != test_df["clean_text"]

train_changed_count = train_df["text_changed_after_cleaning"].sum()
test_changed_count = test_df["text_changed_after_cleaning"].sum()

print("Number of train rows changed after cleaning:", train_changed_count)
print("Number of test rows changed after cleaning:", test_changed_count)

print("\nPercentage of train rows changed:")
print(round((train_changed_count / len(train_df)) * 100, 2), "%")

print("\nPercentage of test rows changed:")
print(round((test_changed_count / len(test_df)) * 100, 2), "%")

print("\nChanged examples from train dataset:")
display(
    train_df.loc[
        train_df["text_changed_after_cleaning"],
        ["original_text", "clean_text", "label", "label_name"]
    ].head(10)
)

print("\nChanged examples from test dataset:")
display(
    test_df.loc[
        test_df["text_changed_after_cleaning"],
        ["original_text", "clean_text", "label", "label_name"]
    ].head(10)
)

In [ ]:
# Check clean_text missing, empty, and whitespace-only values

train_missing_clean_text = train_df["clean_text"].isnull().sum()
test_missing_clean_text = test_df["clean_text"].isnull().sum()

train_empty_clean_text = (train_df["clean_text"] == "").sum()
test_empty_clean_text = (test_df["clean_text"] == "").sum()

train_whitespace_clean_text = train_df["clean_text"].apply(lambda x: str(x).strip() == "").sum()
test_whitespace_clean_text = test_df["clean_text"].apply(lambda x: str(x).strip() == "").sum()

clean_text_quality_summary = pd.DataFrame({
    "split": ["train", "test"],
    "missing_clean_text": [train_missing_clean_text, test_missing_clean_text],
    "empty_clean_text": [train_empty_clean_text, test_empty_clean_text],
    "whitespace_only_clean_text": [train_whitespace_clean_text, test_whitespace_clean_text]
})

print("Clean text quality summary:")
display(clean_text_quality_summary)

In [ ]:
# Check whether cleaning created duplicate clean_text values

train_duplicate_clean_text_count = train_df["clean_text"].duplicated().sum()
test_duplicate_clean_text_count = test_df["clean_text"].duplicated().sum()

print("Duplicate clean_text values in train dataset:", train_duplicate_clean_text_count)
print("Duplicate clean_text values in test dataset:", test_duplicate_clean_text_count)

print("\nDuplicate clean_text examples from train dataset:")
display(
    train_df[train_df["clean_text"].duplicated(keep=False)]
    .sort_values("clean_text")
    [["original_text", "clean_text", "label", "label_name"]]
    .head(20)
)

print("\nDuplicate clean_text examples from test dataset:")
display(
    test_df[test_df["clean_text"].duplicated(keep=False)]
    .sort_values("clean_text")
    [["original_text", "clean_text", "label", "label_name"]]
    .head(20)
)

In [ ]:
# Check whether cleaning created conflicting labels

train_clean_label_counts = train_df.groupby("clean_text")["label"].nunique()
test_clean_label_counts = test_df.groupby("clean_text")["label"].nunique()

train_conflicting_clean_text_count = (train_clean_label_counts > 1).sum()
test_conflicting_clean_text_count = (test_clean_label_counts > 1).sum()

print("Conflicting clean_text labels in train dataset:", train_conflicting_clean_text_count)
print("Conflicting clean_text labels in test dataset:", test_conflicting_clean_text_count)

train_conflicting_clean_texts = train_clean_label_counts[train_clean_label_counts > 1].index
test_conflicting_clean_texts = test_clean_label_counts[test_clean_label_counts > 1].index

print("\nConflicting clean_text examples from train dataset:")
display(
    train_df[train_df["clean_text"].isin(train_conflicting_clean_texts)]
    [["original_text", "clean_text", "label", "label_name"]]
    .sort_values("clean_text")
)

print("\nConflicting clean_text examples from test dataset:")
display(
    test_df[test_df["clean_text"].isin(test_conflicting_clean_texts)]
    [["original_text", "clean_text", "label", "label_name"]]
    .sort_values("clean_text")
)

In [ ]:
#Analyse tokenizer-based token length

from transformers import AutoTokenizer

tokenizer_models = {
    "DistilBERT": "distilbert-base-uncased",
    "RoBERTa": "roberta-base"
}

token_length_summaries = []

for model_name, checkpoint in tokenizer_models.items():
    print(f"\nLoading tokenizer for {model_name}: {checkpoint}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    train_token_lengths = train_df["clean_text"].apply(
        lambda text: len(tokenizer.encode(text, add_special_tokens=True))
    )

    test_token_lengths = test_df["clean_text"].apply(
        lambda text: len(tokenizer.encode(text, add_special_tokens=True))
    )

    combined_token_lengths = pd.concat(
        [train_token_lengths, test_token_lengths],
        ignore_index=True
    )

    summary = {
        "model": model_name,
        "checkpoint": checkpoint,
        "train_min_tokens": train_token_lengths.min(),
        "train_max_tokens": train_token_lengths.max(),
        "train_mean_tokens": round(train_token_lengths.mean(), 2),
        "train_median_tokens": train_token_lengths.median(),
        "test_min_tokens": test_token_lengths.min(),
        "test_max_tokens": test_token_lengths.max(),
        "test_mean_tokens": round(test_token_lengths.mean(), 2),
        "test_median_tokens": test_token_lengths.median(),
        "combined_min_tokens": combined_token_lengths.min(),
        "combined_max_tokens": combined_token_lengths.max(),
        "combined_mean_tokens": round(combined_token_lengths.mean(), 2),
        "combined_median_tokens": combined_token_lengths.median(),
        "combined_over_128": (combined_token_lengths > 128).sum(),
        "combined_over_256": (combined_token_lengths > 256).sum(),
        "combined_over_512": (combined_token_lengths > 512).sum()
    }

    token_length_summaries.append(summary)

token_length_summary_df = pd.DataFrame(token_length_summaries)

print("\nTokenizer-based token length summary:")
display(token_length_summary_df)

In [ ]:
# Decide max_length for transformer fine-tuning

MAX_LENGTH = 512

max_length_decision = pd.DataFrame({
    "decision_item": [
        "Selected max_length",
        "Reason 1",
        "Reason 2",
        "Reason 3",
        "Fine-tuning tokenization setting"
    ],
    "decision": [
        MAX_LENGTH,
        "BERT-style transformer models usually support up to 512 tokens.",
        "Only a very small number of prompts are longer than 512 tokens.",
        "Using 512 preserves more prompt-injection context than 128 or 256.",
        "Use truncation=True, padding='max_length', max_length=512 during model fine-tuning."
    ]
})

print("max_length decision completed.")
display(max_length_decision)

print("\nSelected MAX_LENGTH:", MAX_LENGTH)

In [ ]:
# Keep original test set untouched

final_test_df = test_df.copy()

print("Original test set copied successfully.")
print("This test set will not be used for train/validation splitting.")

print("\nOriginal test dataset shape:")
print(test_df.shape)

print("\nFinal test dataset shape:")
print(final_test_df.shape)

print("\nOriginal test label distribution:")
print(test_df["label_name"].value_counts())

print("\nFinal test label distribution:")
print(final_test_df["label_name"].value_counts())

# Check whether final_test_df is exactly the same as test_df
test_copy_check = final_test_df.equals(test_df)

print("\nfinal_test_df is exactly same as test_df:", test_copy_check)

In [ ]:
# Create validation split from original training set using stratified split

train_split_df, validation_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df["label"]
)

final_train_df = train_split_df.copy()
validation_df = validation_df.copy()

print("Train/validation split created successfully.")

print("\nOriginal training dataset shape:")
print(train_df.shape)

print("\nFinal train dataset shape:")
print(final_train_df.shape)

print("\nValidation dataset shape:")
print(validation_df.shape)

print("\nFinal test dataset shape:")
print(final_test_df.shape)

print("\nOriginal train label distribution:")
print(train_df["label_name"].value_counts())

print("\nFinal train label distribution:")
print(final_train_df["label_name"].value_counts())

print("\nValidation label distribution:")
print(validation_df["label_name"].value_counts())

print("\nFinal test label distribution:")
print(final_test_df["label_name"].value_counts())

In [ ]:
# Verify train/validation/test sizes and class distribution

split_size_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [
        len(final_train_df),
        len(validation_df),
        len(final_test_df)
    ],
    "columns": [
        final_train_df.shape[1],
        validation_df.shape[1],
        final_test_df.shape[1]
    ]
})

print("Split size summary:")
display(split_size_summary)


def get_class_distribution(df, split_name):
    label_counts = df["label_name"].value_counts().reset_index()
    label_counts.columns = ["label_name", "count"]
    label_counts["split"] = split_name
    label_counts["percentage"] = round((label_counts["count"] / len(df)) * 100, 2)
    return label_counts[["split", "label_name", "count", "percentage"]]


train_distribution = get_class_distribution(final_train_df, "train")
validation_distribution = get_class_distribution(validation_df, "validation")
test_distribution = get_class_distribution(final_test_df, "test")

split_class_distribution_summary = pd.concat(
    [train_distribution, validation_distribution, test_distribution],
    ignore_index=True
)

print("\nSplit class distribution summary:")
display(split_class_distribution_summary)

In [ ]:
# Add final split column

final_train_df["split"] = "train"
validation_df["split"] = "validation"
final_test_df["split"] = "test"

print("Final split column added successfully.")

print("\nFinal train columns:")
print(final_train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nFinal test columns:")
print(final_test_df.columns.tolist())

print("\nFinal train split values:")
print(final_train_df["split"].value_counts())

print("\nValidation split values:")
print(validation_df["split"].value_counts())

print("\nFinal test split values:")
print(final_test_df["split"].value_counts())

print("\nFirst 5 rows from final train dataset:")
display(final_train_df[["clean_text", "label", "label_name", "split"]].head())

print("\nFirst 5 rows from validation dataset:")
display(validation_df[["clean_text", "label", "label_name", "split"]].head())

print("\nFirst 5 rows from final test dataset:")
display(final_test_df[["clean_text", "label", "label_name", "split"]].head())

In [ ]:
# Select final model-ready columns

final_columns = [
    "original_text",
    "clean_text",
    "label",
    "label_name",
    "split",
    "text_changed_after_cleaning"
]

final_train_df = final_train_df[final_columns].reset_index(drop=True)
validation_df = validation_df[final_columns].reset_index(drop=True)
final_test_df = final_test_df[final_columns].reset_index(drop=True)

print("Final model-ready columns selected successfully.")

print("\nFinal train columns:")
print(final_train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nFinal test columns:")
print(final_test_df.columns.tolist())

print("\nFinal train shape:")
print(final_train_df.shape)

print("\nValidation shape:")
print(validation_df.shape)

print("\nFinal test shape:")
print(final_test_df.shape)

print("\nFirst 5 rows from final train dataset:")
display(final_train_df.head())

print("\nFirst 5 rows from validation dataset:")
display(validation_df.head())

print("\nFirst 5 rows from final test dataset:")
display(final_test_df.head())

In [ ]:
# Verify final dataset schema

expected_columns = [
    "original_text",
    "clean_text",
    "label",
    "label_name",
    "split",
    "text_changed_after_cleaning"
]

datasets_to_check = {
    "train": final_train_df,
    "validation": validation_df,
    "test": final_test_df
}

schema_results = []

for split_name, df in datasets_to_check.items():
    columns_match = df.columns.tolist() == expected_columns

    schema_results.append({
        "split": split_name,
        "columns_match_expected": columns_match,
        "number_of_rows": len(df),
        "number_of_columns": df.shape[1],
        "column_names": df.columns.tolist(),
        "label_unique_values": sorted(df["label"].unique().tolist()),
        "label_name_unique_values": sorted(df["label_name"].unique().tolist()),
        "split_unique_values": sorted(df["split"].unique().tolist()),
        "clean_text_dtype": str(df["clean_text"].dtype),
        "label_dtype": str(df["label"].dtype)
    })

schema_summary_df = pd.DataFrame(schema_results)

print("Final dataset schema summary:")
display(schema_summary_df)

print("\nExpected columns:")
print(expected_columns)

print("\nFinal train columns match expected:", final_train_df.columns.tolist() == expected_columns)
print("Validation columns match expected:", validation_df.columns.tolist() == expected_columns)
print("Final test columns match expected:", final_test_df.columns.tolist() == expected_columns)

In [ ]:
# Save clean_train.csv

clean_train_path = PROCESSED_DATA_DIR / "clean_train.csv"

final_train_df.to_csv(clean_train_path, index=False, encoding="utf-8")

print("clean_train.csv saved successfully.")

print("\nSaved file path:")
print(clean_train_path)

print("\nFile exists:")
print(clean_train_path.exists())

print("\nSaved file size:")
print(clean_train_path.stat().st_size, "bytes")

print("\nPreview saved clean_train.csv:")
saved_clean_train_df = pd.read_csv(clean_train_path)
display(saved_clean_train_df.head())

print("\nSaved clean_train.csv shape:")
print(saved_clean_train_df.shape)

print("\nSaved clean_train.csv columns:")
print(saved_clean_train_df.columns.tolist())

In [ ]:
# Save clean_validation.csv

clean_validation_path = PROCESSED_DATA_DIR / "clean_validation.csv"

validation_df.to_csv(clean_validation_path, index=False, encoding="utf-8")

print("clean_validation.csv saved successfully.")

print("\nSaved file path:")
print(clean_validation_path)

print("\nFile exists:")
print(clean_validation_path.exists())

print("\nSaved file size:")
print(clean_validation_path.stat().st_size, "bytes")

print("\nPreview saved clean_validation.csv:")
saved_clean_validation_df = pd.read_csv(clean_validation_path)
display(saved_clean_validation_df.head())

print("\nSaved clean_validation.csv shape:")
print(saved_clean_validation_df.shape)

print("\nSaved clean_validation.csv columns:")
print(saved_clean_validation_df.columns.tolist())

In [ ]:
# Save clean_test.csv

clean_test_path = PROCESSED_DATA_DIR / "clean_test.csv"

final_test_df.to_csv(clean_test_path, index=False, encoding="utf-8")

print("clean_test.csv saved successfully.")

print("\nSaved file path:")
print(clean_test_path)

print("\nFile exists:")
print(clean_test_path.exists())

print("\nSaved file size:")
print(clean_test_path.stat().st_size, "bytes")

print("\nPreview saved clean_test.csv:")
saved_clean_test_df = pd.read_csv(clean_test_path)
display(saved_clean_test_df.head())

print("\nSaved clean_test.csv shape:")
print(saved_clean_test_df.shape)

print("\nSaved clean_test.csv columns:")
print(saved_clean_test_df.columns.tolist())

In [ ]:
# Save optional parquet versions of processed datasets

clean_train_parquet_path = PROCESSED_DATA_DIR / "clean_train.parquet"
clean_validation_parquet_path = PROCESSED_DATA_DIR / "clean_validation.parquet"
clean_test_parquet_path = PROCESSED_DATA_DIR / "clean_test.parquet"

final_train_df.to_parquet(clean_train_parquet_path, index=False)
validation_df.to_parquet(clean_validation_parquet_path, index=False)
final_test_df.to_parquet(clean_test_parquet_path, index=False)

print("Processed parquet files saved successfully.")

print("\nSaved parquet files:")
print(clean_train_parquet_path)
print(clean_validation_parquet_path)
print(clean_test_parquet_path)

print("\nFile existence check:")
print("clean_train.parquet exists:", clean_train_parquet_path.exists())
print("clean_validation.parquet exists:", clean_validation_parquet_path.exists())
print("clean_test.parquet exists:", clean_test_parquet_path.exists())

print("\nFile sizes:")
print("clean_train.parquet:", clean_train_parquet_path.stat().st_size, "bytes")
print("clean_validation.parquet:", clean_validation_parquet_path.stat().st_size, "bytes")
print("clean_test.parquet:", clean_test_parquet_path.stat().st_size, "bytes")

print("\nReload parquet files to verify:")
saved_train_parquet_df = pd.read_parquet(clean_train_parquet_path)
saved_validation_parquet_df = pd.read_parquet(clean_validation_parquet_path)
saved_test_parquet_df = pd.read_parquet(clean_test_parquet_path)

print("Saved train parquet shape:", saved_train_parquet_df.shape)
print("Saved validation parquet shape:", saved_validation_parquet_df.shape)
print("Saved test parquet shape:", saved_test_parquet_df.shape)

print("\nSaved train parquet columns:")
print(saved_train_parquet_df.columns.tolist())

In [ ]:
# Save processed dataset backup to Google Drive

processed_files_to_backup = [
    "clean_train.csv",
    "clean_validation.csv",
    "clean_test.csv",
    "clean_train.parquet",
    "clean_validation.parquet",
    "clean_test.parquet"
]

backup_results = []

for file_name in processed_files_to_backup:
    source_path = PROCESSED_DATA_DIR / file_name
    destination_path = DRIVE_PROCESSED_DATA_DIR / file_name

    if source_path.exists():
        shutil.copy2(source_path, destination_path)

        backup_results.append({
            "file_name": file_name,
            "source_exists": source_path.exists(),
            "backup_exists": destination_path.exists(),
            "source_path": str(source_path),
            "backup_path": str(destination_path),
            "backup_file_size_bytes": destination_path.stat().st_size
        })
    else:
        backup_results.append({
            "file_name": file_name,
            "source_exists": False,
            "backup_exists": False,
            "source_path": str(source_path),
            "backup_path": str(destination_path),
            "backup_file_size_bytes": 0
        })

backup_summary_df = pd.DataFrame(backup_results)

print("Processed dataset backup to Google Drive completed.")

print("\nBackup summary:")
display(backup_summary_df)

print("\nFiles inside Google Drive processed data folder:")
for file in DRIVE_PROCESSED_DATA_DIR.iterdir():
    print(file.name)

In [ ]:
# Save final split summary table

final_split_size_summary_path = TABLES_DIR / "final_split_size_summary.csv"
final_split_class_distribution_path = TABLES_DIR / "final_split_class_distribution_summary.csv"

drive_split_size_summary_path = DRIVE_TABLES_DIR / "final_split_size_summary.csv"
drive_split_class_distribution_path = DRIVE_TABLES_DIR / "final_split_class_distribution_summary.csv"

split_size_summary.to_csv(final_split_size_summary_path, index=False, encoding="utf-8")
split_class_distribution_summary.to_csv(final_split_class_distribution_path, index=False, encoding="utf-8")

shutil.copy2(final_split_size_summary_path, drive_split_size_summary_path)
shutil.copy2(final_split_class_distribution_path, drive_split_class_distribution_path)

print("Final split summary tables saved successfully.")

print("\nLocal saved files:")
print(final_split_size_summary_path)
print(final_split_class_distribution_path)

print("\nGoogle Drive backup files:")
print(drive_split_size_summary_path)
print(drive_split_class_distribution_path)

print("\nFile existence check:")
print("Local split size summary exists:", final_split_size_summary_path.exists())
print("Local class distribution summary exists:", final_split_class_distribution_path.exists())
print("Drive split size summary exists:", drive_split_size_summary_path.exists())
print("Drive class distribution summary exists:", drive_split_class_distribution_path.exists())

print("\nFinal split size summary:")
display(pd.read_csv(final_split_size_summary_path))

print("\nFinal split class distribution summary:")
display(pd.read_csv(final_split_class_distribution_path))

In [ ]:
# Save final cleaning summary table

cleaning_summary = pd.DataFrame({
    "item": [
        "Cleaning strategy",
        "Aggressive cleaning used",
        "Original text preserved",
        "Clean text column created",
        "Train rows changed after cleaning",
        "Test rows changed after cleaning",
        "Train missing clean_text",
        "Test missing clean_text",
        "Train empty clean_text",
        "Test empty clean_text",
        "Train whitespace-only clean_text",
        "Test whitespace-only clean_text",
        "Train duplicate clean_text",
        "Test duplicate clean_text",
        "Train conflicting clean_text labels",
        "Test conflicting clean_text labels",
        "URLs removed",
        "Code snippets removed",
        "Markdown-style text removed",
        "HTML-like text removed",
        "Unicode/multilingual text removed",
        "Rows deleted during cleaning"
    ],
    "value": [
        "Light cleaning only",
        "No",
        "Yes",
        "Yes",
        train_changed_count,
        test_changed_count,
        train_missing_clean_text,
        test_missing_clean_text,
        train_empty_clean_text,
        test_empty_clean_text,
        train_whitespace_clean_text,
        test_whitespace_clean_text,
        train_duplicate_clean_text_count,
        test_duplicate_clean_text_count,
        train_conflicting_clean_text_count,
        test_conflicting_clean_text_count,
        "No",
        "No",
        "No",
        "No",
        "No",
        "No"
    ]
})

cleaning_summary_path = TABLES_DIR / "final_cleaning_summary.csv"
drive_cleaning_summary_path = DRIVE_TABLES_DIR / "final_cleaning_summary.csv"

cleaning_summary.to_csv(cleaning_summary_path, index=False, encoding="utf-8")
shutil.copy2(cleaning_summary_path, drive_cleaning_summary_path)

print("Final cleaning summary table saved successfully.")

print("\nLocal saved file:")
print(cleaning_summary_path)

print("\nGoogle Drive backup file:")
print(drive_cleaning_summary_path)

print("\nFile existence check:")
print("Local cleaning summary exists:", cleaning_summary_path.exists())
print("Drive cleaning summary exists:", drive_cleaning_summary_path.exists())

print("\nFinal cleaning summary:")
display(pd.read_csv(cleaning_summary_path))

In [ ]:
# Save final token-length summary table

token_length_summary_path = TABLES_DIR / "final_token_length_summary.csv"
drive_token_length_summary_path = DRIVE_TABLES_DIR / "final_token_length_summary.csv"

token_length_summary_df["selected_max_length"] = MAX_LENGTH
token_length_summary_df["fine_tuning_truncation_required"] = token_length_summary_df["combined_over_512"] > 0

token_length_summary_df.to_csv(token_length_summary_path, index=False, encoding="utf-8")

shutil.copy2(token_length_summary_path, drive_token_length_summary_path)

print("Final token-length summary table saved successfully.")

print("\nLocal saved file:")
print(token_length_summary_path)

print("\nGoogle Drive backup file:")
print(drive_token_length_summary_path)

print("\nFile existence check:")
print("Local token-length summary exists:", token_length_summary_path.exists())
print("Drive token-length summary exists:", drive_token_length_summary_path.exists())

print("\nFinal token-length summary:")
display(pd.read_csv(token_length_summary_path))

In [ ]:
# Check final processed files exist

final_processed_files = [
    "clean_train.csv",
    "clean_validation.csv",
    "clean_test.csv",
    "clean_train.parquet",
    "clean_validation.parquet",
    "clean_test.parquet"
]

file_check_results = []

for file_name in final_processed_files:
    file_path = PROCESSED_DATA_DIR / file_name

    file_check_results.append({
        "file_name": file_name,
        "file_exists": file_path.exists(),
        "file_path": str(file_path),
        "file_size_bytes": file_path.stat().st_size if file_path.exists() else 0
    })

final_processed_file_check_df = pd.DataFrame(file_check_results)

print("Final processed file check:")
display(final_processed_file_check_df)

print("\nAll final processed files exist:")
print(final_processed_file_check_df["file_exists"].all())

In [ ]:
# final processed file shapes
processed_file_paths = {
    "clean_train.csv": PROCESSED_DATA_DIR / "clean_train.csv",
    "clean_validation.csv": PROCESSED_DATA_DIR / "clean_validation.csv",
    "clean_test.csv": PROCESSED_DATA_DIR / "clean_test.csv",
    "clean_train.parquet": PROCESSED_DATA_DIR / "clean_train.parquet",
    "clean_validation.parquet": PROCESSED_DATA_DIR / "clean_validation.parquet",
    "clean_test.parquet": PROCESSED_DATA_DIR / "clean_test.parquet"
}

processed_shape_results = []

for file_name, file_path in processed_file_paths.items():
    if file_name.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_name.endswith(".parquet"):
        df = pd.read_parquet(file_path)

    processed_shape_results.append({
        "file_name": file_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": df.columns.tolist()
    })

final_processed_shape_summary_df = pd.DataFrame(processed_shape_results)

print("Final processed file shape summary:")
display(final_processed_shape_summary_df)

In [ ]:
# Check final processed class distribution again

final_processed_class_distribution_results = []

for file_name, file_path in processed_file_paths.items():
    if file_name.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_name.endswith(".parquet"):
        df = pd.read_parquet(file_path)

    label_counts = df["label_name"].value_counts().reset_index()
    label_counts.columns = ["label_name", "count"]
    label_counts["file_name"] = file_name
    label_counts["percentage"] = round((label_counts["count"] / len(df)) * 100, 2)

    final_processed_class_distribution_results.append(
        label_counts[["file_name", "label_name", "count", "percentage"]]
    )

final_processed_class_distribution_df = pd.concat(
    final_processed_class_distribution_results,
    ignore_index=True
)

print("Final processed file class distribution:")
display(final_processed_class_distribution_df)

In [ ]:
# Save final preparation report

REPORTS_DIR = RESULTS_DIR / "reports"
DRIVE_REPORTS_DIR = DRIVE_PROJECT_DIR / "results" / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

final_preparation_report_path = REPORTS_DIR / "final_dataset_preparation_report.md"
drive_final_preparation_report_path = DRIVE_REPORTS_DIR / "final_dataset_preparation_report.md"

report_text = f"""
# Final Dataset Preparation Report

## Project

Prompt Injection Detection Using Fine-Tuned Transformer Models: A Glass-Box Approach with Explainable AI.

## Dataset

The dataset used in this stage is the deepset prompt-injections dataset. The original dataset was loaded from raw parquet files and prepared for binary text classification.

## Label Mapping

- `0` = SAFE
- `1` = INJECTION

## Original Dataset Size

- Original train set: {train_df.shape[0]} rows
- Original test set: {test_df.shape[0]} rows
- Total dataset size: {train_df.shape[0] + test_df.shape[0]} rows

## Cleaning Strategy

The dataset preparation used light text cleaning only. Aggressive cleaning was not applied because prompt-injection signals may appear in punctuation, casing, symbols, URLs, code snippets, markdown-style text, HTML-like text, multilingual text, and command-style phrases.

## Cleaning Rules Applied

1. The original text was preserved in the `original_text` column.
2. A cleaned version was created in the `clean_text` column.
3. Leading and trailing spaces were removed.
4. Repeated whitespace, tabs, and new lines were replaced with a single space.
5. Punctuation and symbols were preserved.
6. Capitalisation was preserved.
7. URLs were preserved.
8. Code snippets were preserved.
9. Markdown-style and HTML-like text were preserved.
10. Unicode and multilingual text were preserved.
11. No rows were deleted during cleaning.

## Cleaning Results

- Train rows changed after cleaning: {train_changed_count}
- Test rows changed after cleaning: {test_changed_count}
- Train missing clean_text values: {train_missing_clean_text}
- Test missing clean_text values: {test_missing_clean_text}
- Train empty clean_text values: {train_empty_clean_text}
- Test empty clean_text values: {test_empty_clean_text}
- Train whitespace-only clean_text values: {train_whitespace_clean_text}
- Test whitespace-only clean_text values: {test_whitespace_clean_text}
- Train duplicate clean_text values: {train_duplicate_clean_text_count}
- Test duplicate clean_text values: {test_duplicate_clean_text_count}
- Train conflicting clean_text labels: {train_conflicting_clean_text_count}
- Test conflicting clean_text labels: {test_conflicting_clean_text_count}

## Final Dataset Splits

- Final train set: {final_train_df.shape[0]} rows
- Validation set: {validation_df.shape[0]} rows
- Final test set: {final_test_df.shape[0]} rows

## Final Train Class Distribution

{final_train_df["label_name"].value_counts().to_string()}

## Validation Class Distribution

{validation_df["label_name"].value_counts().to_string()}

## Final Test Class Distribution

{final_test_df["label_name"].value_counts().to_string()}

## Token-Length Decision

Tokenizer-based analysis was completed using DistilBERT and RoBERTa tokenizers.

The selected maximum sequence length for fine-tuning is:

- `max_length = {MAX_LENGTH}`

This value was selected because BERT-style transformer models usually support up to 512 tokens, and only a very small number of prompts exceeded 512 tokens.

During fine-tuning, tokenization should use:

- `truncation=True`
- `padding="max_length"`
- `max_length=512`

## Final Processed Files

The following processed dataset files were saved:

1. `clean_train.csv`
2. `clean_validation.csv`
3. `clean_test.csv`
4. `clean_train.parquet`
5. `clean_validation.parquet`
6. `clean_test.parquet`

The processed files were saved locally in:

`{PROCESSED_DATA_DIR}`

The processed files were also backed up to Google Drive in:

`{DRIVE_PROCESSED_DATA_DIR}`

## Final Model-Ready Columns

Each final processed dataset contains the following columns:

1. `original_text`
2. `clean_text`
3. `label`
4. `label_name`
5. `split`
6. `text_changed_after_cleaning`

## Fine-Tuning Usage

For model fine-tuning, use:

- Input column: `clean_text`
- Target column: `label`

The `original_text` column is preserved for checking, reporting, debugging, and later explainability comparison.
"""

with open(final_preparation_report_path, "w", encoding="utf-8") as file:
    file.write(report_text)

shutil.copy2(final_preparation_report_path, drive_final_preparation_report_path)

print("Final dataset preparation report saved successfully.")

print("\nLocal report path:")
print(final_preparation_report_path)

print("\nGoogle Drive backup report path:")
print(drive_final_preparation_report_path)

print("\nFile existence check:")
print("Local report exists:", final_preparation_report_path.exists())
print("Drive report exists:", drive_final_preparation_report_path.exists())

print("\nReport preview:")
with open(final_preparation_report_path, "r", encoding="utf-8") as file:
    preview_text = file.read()

print(preview_text[:2000])